# UNOCHA Humanitarian Funding Gap Analysis — Handoff Prompt

## Project Goal
Build a humanitarian funding gap analysis pipeline on Databricks to identify
"overlooked" crises — situations with high human need but disproportionately
low funding. The pipeline serves a RAG agent that answers questions like
"which crises are most overlooked?" and "show me food security hotspots
with <10% funding."

---

## Pipeline Architecture

```
Bronze (raw ingestion)
    ↓
Silver (cleaned/normalised, 23 tables)   ← COMPLETE
    ↓
Gold (5 analytical tables)               ← COMPLETE & VALIDATED
    ↓
[Upstream application layer]             ← THIS SESSION'S FOCUS
```

**Workspace coordinates:**
- Catalog/Schema: `workspace.default`
- Gold prefix: `njyoti_gold_`
- Silver prefix: `njyoti_silver_unocha_`
- Notebook: `/Users/nikita.j26@gmail.com/Gold_layer` (ID: 876858178797360)

---

## Gold Layer — Complete & Validated

### Table 1: `njyoti_gold_country_year_funding`
- Grain: `iso3 × year` | 1,120 rows | 126 countries | 2000–2026
- Key columns: `fts_requirements_usd`, `fts_funding_usd`, `fts_percent_funded`,
  `funding_gap_usd`, `funding_gap_pct`, `inform_severity_index`,
  `severity_trend_label`, `years_below_threshold`, `yoy_pct_funded_change`,
  `total_allocations_usd`, `total_population`, `per_capita_funding_usd`,
  `data_sources`, `source_count`
- Sources joined: FTS + HRP + INFORM severity + INFORM trends + COD population
  + CERF/CBPF allocations + crisis registry + C&T Taxonomy (country name fallback)

### Table 2: `njyoti_gold_sector_funding_gaps`
- Grain: `iso3 × year × sector_canonical` | 668 rows | 24 countries | 2024–2026
- Key columns: `sector_canonical`, `sector_code`, `sector_name`, `in_need`,
  `targeted`, `sector_requirements_usd`, `sector_funding_usd`,
  `sector_pct_funded`, `sector_funding_gap_usd`, `coverage_ratio`,
  `fts_cluster_names_matched`
- Match rate: **95.4%** (up from 67% regex — fixed via OCHA taxonomy bridge join)
- All Protection sub-clusters now matched: Child Protection 100%, GBV 94.1%, Mine Action 87.5%, HLP 79.2%

### Table 3: `njyoti_gold_overlooked_crises_scored`
- Grain: `iso3 × year` | 377 rows | 71 countries | 31 columns (scorable universe — 43.2% of total FTS requirements are unscored)
- Map-ready: `latitude`, `longitude` (country centroid, WGS84) denormalized from C&T Taxonomy. 100% coverage, zero NULLs.
- Scoring formula (multiplicative):
  ```
  overlooked_score = severity_norm          [INFORM index / 5.0]
                   × funding_gap_norm       [CLAMP(1 - pct_funded/100, 0, 1)]
                   × scale_norm             [CLAMP((log10(req) - 5) / 5, 0, 1)]
                   × trend_multiplier       [Increasing=1.3, Stable=1.0, Decreasing=0.8]
                   × persistence_multiplier [1.0 + (yrs_below_threshold/5) × 0.5]
                   × allocation_neglect     [1 - MIN(1, allocations/requirements)]
  ```
- `rank_in_year`: `DENSE_RANK()` partitioned by `year`, ordered by `overlooked_score DESC`
- Validation: P90/P10 = 6.5x | 7/8 known severe crises in top 20 (AFG, YEM, SSD, COD, SOM, SYR, MMR — PRK ranks 8th but `inform_tracked=False`)

### Table 4: `njyoti_gold_regional_summary`
- Grain: `region × year` | 112 rows | 5 regions (Africa, Americas, Asia, Europe, Middle East)
- Aggregated from Table 1. Cross-table consistency: 0.00% diff vs Table 1 totals.

### Table 5: `njyoti_gold_inform_temporal_series`
- Grain: `crisis_id × year_month` | 6,889 rows | 125 crises | 2019-01 to 2026-04
- Severity time series for RAG drill-down and trajectory visualisation.
  Deduped from 251,008 rows (each snapshot had full history).

---

## Query Pattern → Table Routing

The RAG agent should route queries to gold tables as follows:

| User query type | Primary table | Drill-down |
|---|---|---|
| "Most overlooked crises" / ranking | T3 `overlooked_crises_scored` | T5 (trajectory), T2 (by sector) |
| "Country X funding gap this year" | T1 `country_year_funding` | T2 (by sector) |
| "Food security / WASH / Health hotspots" | T2 `sector_funding_gaps` | T1 (country context) |
| "Which region is most underfunded" | T4 `regional_summary` | T1 (country detail) |
| "Has crisis X been getting worse" | T5 `inform_temporal_series` | T3 (score rank) |
| "Conflict-driven vs climate-driven crises" | T3 (`is_conflict_driven`, `is_climate_driven` flags) | T1 (funding) |
| "Which crises have been underfunded for years" | T3 (`years_below_threshold`, `persistence_multiplier`) | T1 |

---

## Limitations the RAG Agent Must Surface

These are hard constraints on what the gold layer can and cannot answer.
Any agent response touching these areas **must** include a caveat.

1. **43.2% of FTS requirements are unscored.** Table 3 has 377 rows vs 1,120 in Table 1. A country with FTS data but no INFORM severity score cannot be ranked. Responses to "most overlooked" must note this covers ~57% of total stated requirements.

2. **17 countries scored without INFORM Crisis Registry tracking.** The `inform_tracked` boolean column in Table 3 flags these countries. The 17 iso3 codes are: PRK, ZWE, ZMB, COG, BOL, VNM, NPL, RWA, MNG, LSO (crisis-adjacent) + POL, MDA, CZE, ROU, BGR, LTU, EST (Ukraine refugee-hosting EU/EE). These countries DO have real INFORM severity scores (from the all_crises assessment table), but are absent from the Crisis Registry — meaning they lack `country_name`, `crisis_drivers`, `severity_trend_label` (defaults to trend_multiplier=1.0), and `region` assignment. Severity data is present and valid; only registry metadata is missing.

3. **Bilateral aid excluded.** `fts_percent_funded` counts only funding through FTS-tracked plans. Real coverage may be higher.

4. **Sector coverage.** 95.4% of sector-country-year rows now have FTS funding data via OCHA taxonomy join. All 19 HNO sectors including Protection sub-clusters are matched. The ~5% unmatched rows are genuine gaps (no FTS globalcluster entry for that combination).

5. **2026 severity values are provisional.** OCHA changed the INFORM scale from 1–5 to 1–10 in 2026; gold layer normalises by dividing by 2, introducing uncertainty for 2026 rankings.

6. **CERF/CBPF allocations cover only 35% of scored rows.** The `allocation_neglect` component defaults to 1.0 (maximum neglect) for the other 65%, biasing scores upward for countries with missing allocations data.

---

## Validation Status (Final — all checks pass, C-reliability failures = 0)

| Check | Status | Notes |
|---|---|---|
| B1 Score discrimination | ✅ | P90/P10 = 6.5x |
| B2 Face validity | ✅ | 7/8 known severe crises in top 20 |
| B3 Sector join | ✅ | 95.4% match (via taxonomy join) |
| B4 Temporal features | ✅ | 100% persistence, 87% trend |
| B5 Allocation signal | ✅ | 69% at max neglect |
| C1 Referential integrity | ✅ | 17 INFORM-untracked iso3 confirmed in C&T Taxonomy — informational only |
| C2 Value ranges | ✅ | Severity [1.1, 5.0], all norms [0, 1] |
| C3 Grain duplicates | ✅ | 0 duplicates in all tables |
| C4 Cross-table consistency | ✅ | 0.00% diff |
| C5 Temporal coverage | ✅ | 97% avg coverage |
| C6 Spot-check | ✅ | AFG 2026 matches silver exactly |
| C7 Data loss | ✅ | No country loss |
| HNO fan-out | ✅ | `coverage_ratio > 1.0` = 0 rows (fixed from 3); `in_need` now matches HNO national totals |

---

## Locked Design Decisions (Do Not Revisit)

- `UNDERFUNDED_THRESHOLD = 33%` — calibrated from P25–P33 empirical band
- Multiplicative score — intersection of all factors required for high rank
- 5-table gold structure — including temporal series for agent drill-down
- Severity normalised to 1–5 scale at Table 1 (2026 values ÷2)
- Static population join (`iso3` only, no year) — one COD figure per country
- Left joins outward from FTS/HRP spine — preserves all funding data
- `PERSISTENCE_LOOKBACK_YEARS = 5`
- `TREND_MULTIPLIER_MAP = {Increasing: 1.3, Stable: 1.0, Decreasing: 0.8, -/NULL: 1.0}`

---

## Key Assumptions

| ID | Assumption |
|---|---|
| A1 | FTS requirements = stated need (politically constructed; over/under-appeal possible) |
| A3 | Absence from INFORM Crisis Registry ≠ absence of severity data — 17 countries have real INFORM severity scores from all_crises but lack registry metadata (crisis_drivers, trend_label defaults to neutral) |
| A5 | COD population: `population_group="T_TL"`, `gender="all"`, `admin_level=0` |
| A6 | `fts_percent_funded` = same-plan funding only; bilateral aid excluded |
| A10 | HNO covers 2024–2026 only; scoring most reliable 2020+ |

---

## Key Silver Tables (for reference)

| Table | Purpose |
|---|---|
| `njyoti_silver_unocha_fts_requirements_funding_global` | Core funding gap source |
| `njyoti_silver_unocha_fts_requirements_funding_globalcluster_global` | Sector-level FTS funding via taxonomy IDs (USED for Table 2) |
| `njyoti_silver_unocha_hpc_global_cluster_taxonomy` | Bridge table: cluster_code ↔ cluster_id ↔ canonical name (22 rows) |
| `njyoti_silver_unocha_inform_crisis_registry` | iso3 bridge, region, crisis drivers |
| `njyoti_silver_unocha_inform_all_crises` | Severity index/category by snapshot |
| `njyoti_silver_unocha_inform_trends` | Wide-format severity time series (y2019_01…y2026_04) |
| `njyoti_silver_unocha_hpc_hno` | HNO sector needs (`cluster`=code, `description`=free-text) |
| `njyoti_silver_unocha_cod_population` | COD population (`population_group="T_TL"`, `admin_level=0`) |
| `njyoti_silver_unocha_allocations` | CERF/CBPF allocations |
| `njyoti_silver_unocha_humanitarian_response_plans` | HRP plan requirements |
| `njyoti_silver_unocha_country_territory_taxonomy` | C&T Taxonomy MVP — authoritative iso3→country_name fallback (`admin_level=0` filter) |

---

## External Data Sources

Two reference datasets were ingested from external OCHA sources to serve as bridge/lookup tables:

| Source | URL | Silver Table | Purpose |
|---|---|---|---|
| OCHA Countries & Territories | https://data.humdata.org/dataset/countries-and-territories | `njyoti_silver_unocha_country_territory_taxonomy` | Reference table mapping ISO3 codes to country names, regions, sub-regions, coordinates, income levels. Authoritative fallback for iso3→country_name (covers 17 orphan codes absent from INFORM registry). |
| HPC Global Cluster Taxonomy | https://api.hpc.tools/v1/public/global-cluster | `njyoti_silver_unocha_hpc_global_cluster_taxonomy` | Bridge table mapping cluster codes ↔ numeric IDs ↔ canonical English names (22 rows). Enables the taxonomy join path: HNO.cluster = taxonomy.cluster_code → taxonomy.cluster_id = CAST(FTS.clustercode AS INT). |

Both were ingested in the Bronze layer (Phase 7 notebook, ID: 1792158697225031) and are used across gold table transformations.

---

## Minor Open Items from Gold Layer

These are low-priority and can be addressed in parallel or deferred:

1. **`inform_tracked` boolean in Table 3** — DONE. Column added, 17 iso3 codes marked FALSE.
2. **Config cell comment cosmetic fix** — A5 comment still says `population_group="all"` but code correctly uses `"T_TL"`. No functional impact.
3. **WASH sector match** — RESOLVED (98.2% via taxonomy join).

---

## Documentation

A markdown cell (cell 2) in the Gold_layer notebook documents:
- Full gap formula and column definitions
- Overlooked score formula with component table (formula, range, interpretation)
- Sector-level gap formulas (Table 2)
- All limitations: scoring universe (43.2% unscored), severity data (2026 /2 normalisation, country-level trend), funding data (bilateral aid exclusion, 35% allocations coverage), population (static denominator, 82% coverage)
- Known issues table (8 entries with status, including HNO fan-out — fixed)
- Failure cases & edge behaviour: score=0 logic, rank ties, PRK caveat, Americas structural pattern, YEM 2024 Multi-Sector genuine over-target

---

## This Session's Focus: Application Layer Design

The gold layer is production-ready. Design the layer(s) that consume it.
**Choose a focus and proceed — do not ask for more context, everything you need is above.**

**Option A — RAG agent (Genie Space)**
Set up a Genie Space over the 5 gold tables. Define:
- Which tables to expose and with what descriptions
- Natural-language question → SQL patterns for each query type in the routing table above
- How to surface the 6 required caveats in Genie responses
- How to handle multi-table answers (e.g. rank from T3 + sector detail from T2 + trajectory from T5)

**Option B — Analytical dashboard (Lakeview)**
Build a Lakeview dashboard with:
- Top-N overlooked crises panel (T3, filterable by year + region)
- Regional funding trends (T4, multi-year line chart)
- Sector hotspot heatmap (T2, sector × country, coloured by `sector_pct_funded`)
- Severity trajectory (T5, sparklines or line chart per crisis)
- Drill-through path: region → country → sector

**Option C — Both (dashboard first)**
Dashboard provides human-readable validation of the data before the agent goes live. Build Option B first, use it to spot any remaining data issues, then design Option A on a clean foundation.

**Recommended first question to the user:**
Which option? If RAG agent — Genie Space, LangChain chain, or Model Serving endpoint?
If dashboard — audience (internal analysts, external donors, operational teams)?

## Recommended Design for V1

###Phase 1 — Genie Space (1–2 sessions)

Create a Genie Space over all 5 gold tables
Write rich table/column descriptions mapping to your query routing table
Add general instructions encoding the 6 caveats as conditional guardrails
Add sample questions for each query pattern
First add the inform_tracked column to Table 3 (one-liner ALTER TABLE)

**`Phase 1 was powerful enough for the solution. Thus, Phase 2 and 3 were not implemented.`**

###Phase 2 — Lakeview Dashboard (1 session, can parallel Phase 1)

Build the dashboard from your handoff Option B
Use it as human validation that Genie answers match visual reality
Serves as the "donor-facing" output independent of the agent

###Phase 3 — Supervisor + Vector Search (only if needed)

Only pursue if Genie alone can't handle multi-table synthesis queries
Only add Vector Search when/if you ingest qualitative OCHA reports
Lakebase for memory only if user testing reveals multi-turn friction

   
## Genie Space (RAG)

**Space:** UNOCHA Humanitarian Funding Gap Analysis  
**ID:** `01f1565ae8131dd19ee95ddc6c88dbb1`  
**Tables:** 5 gold tables (`workspace.default.njyoti_gold_*`)  
**Run-as:** VIEWER

---

### Configuration Log

#### Phase A — Foundation Setup (Complete)

**1. `inform_tracked` column added to Table 3**
- ALTER TABLE added BOOLEAN column to `njyoti_gold_overlooked_crises_scored`
- 17 iso3 codes marked FALSE (PRK, ZWE, ZMB, COG, BOL, VNM, NPL, RWA, MNG, LSO, POL, MDA, CZE, ROU, BGR, LTU, EST)
- 50 rows affected (13.3% of table), 327 rows TRUE (86.7%)

**2. General Instructions**
- Space description with data coverage summary
- 6 conditional caveats as guardrails (scoring universe, INFORM-untracked, bilateral aid, sector gaps, 2026 provisional, allocation bias)
- Query routing guidelines mapping question types → tables

**3. Table Descriptions** (all 5 tables)
- Each includes: grain, row count, coverage scope, example queries

**4. Starter Questions** (9 total)
- "Which crises are most overlooked in 2025?"
- "Show me food security funding gaps by country"
- "What is Yemen's funding history?"
- "Which region is most underfunded?"
- "Has the Somalia crisis been getting worse?"
- "Show countries underfunded for more than 3 consecutive years"
- "Compare sector funding gaps in Sudan vs Ethiopia"
- "What are the top 10 overlooked crises where inform_tracked is true?"
- "Show Protection sub-cluster funding gaps (Child Protection, GBV, HLP, Mine Action)"

**5. Entity Matching** (12 columns indexed)
- `region` — all 5 tables
- `country_name` — Tables 1, 3, 5
- `sector_canonical` — Table 2
- `severity_trend_label` — Tables 1, 3
- `trend` — Table 5

**6. Hidden Columns** (5 internal columns removed from Genie view)
- `fts_plan_codes` (Table 1)
- `fts_cluster_names_matched` (Table 2)
- `sector_code` (Table 2)
- `snapshot_month` (Table 5)
- `is_2026_snapshot` (Table 5)

**7. Column Synonyms** (47 mappings across all 5 tables)
- Business terms mapped: funding gap, shortfall, underfunding, neglected, CERF neglect, severity, people in need, beneficiaries, worsening, conflict-driven, climate-driven, etc.
- Enables natural language like "show conflict-driven crises in Africa that are getting worse"

---

### Remaining Improvement Phases

#### Phase B — Structural Improvements (Complete)

**Evaluation methodology:** Generated 10 realistic user questions, translated to SQL, executed, then evaluated each question+result pair from a blind user perspective rating Relevance, Accuracy, Correctness (1-5 scale). Baseline average: 4.17/5. Two critical structural failures identified (Q5: 3.3 avg, Q8: 2.3 avg).

**8. Text Instructions Added** (8 instructions)
- Crisis context: Always include `crisis_drivers` and `tracked_crises_count` with scored results
- Cross-table T3→T2: Must use LEFT JOIN (Table 2 only covers 24 countries)
- Cross-table T5→T1: Must deduplicate T5 first (multiple crisis_id per country causes fan-out)
- NULL sector funding: Explain as "data unavailable" not "zero gap"
- NULL allocations: Explain as "not ingested" not "zero CERF/CBPF"
- Temporal scope: "Last N years" = current_year minus N through 2026
- Temporal scope: Table 2 only covers 2024-2026
- Response pattern: Lead with the direct answer, then supporting data

**9. SQL Examples Added** (4 query patterns)
- LEFT JOIN T3→T2: Top overlooked crises with sector breakdown
- Deduplicated T5→T1: Severity trajectory vs funding coverage (ROW_NUMBER pattern)
- Crisis-context ranking: Top overlooked with crisis_drivers included
- Persistence + worsening: Compound filter with full context columns

**10. Join Snippets Added** (4 relationships)
- T3 → T2 (one_to_many, iso3+year, LEFT JOIN required)
- T1 → T3 (one_to_one, iso3+year)
- T3 → T5 (one_to_many, iso3, deduplicate T5 first)
- T1 → T2 (one_to_many, iso3+year, LEFT JOIN required)

**11. Filter Snippets Added** (3 reusable filters)
- Sectors with funding data: `sector_requirements_usd IS NOT NULL`
- Confirmed allocation data: `total_allocations_usd IS NOT NULL`
- INFORM-tracked only: `inform_tracked = TRUE`

**12. Derived Measures Added** (3 snippets)
- Funding gap in billions (human-readable)
- Total funding gap across countries (SUM aggregate)
- Average overlooked score (AVG aggregate)

**13. Additional Synonyms** (crisis context)
- `crisis_drivers`: crisis name, which crisis, crisis type, what crisis, drivers, causes
- `tracked_crises_count`: number of crises, how many crises, crisis count
- `crisis_id` (T5): crisis, crisis name, INFORM crisis, which crisis

---

### Improvement Phases

#### Phase C — Quality Measurement 
- [x] Added 12 evaluation questions as benchmarks with expected SQL (10 original + 2 sub-cluster: Child Protection gaps by country, Protection sub-cluster comparison)
- [ ] Ran benchmarks and measure post-improvement scores
- [ ] Iterated on instructions based on failure analysis

#### Phase D — Negative Testing


Tested off topic messages and Genie handled them gracefully by reiterating it's purpose and not engaging in the topic.


---

### Known Gaps & Structural Limitations

#### 17 ISO3 codes untracked by INFORM (persists after crisis registry ingestion)

The `njyoti_silver_unocha_inform_crisis_registry` (Lists sheet, 127 rows) reflects INFORM's **active monitoring scope** — it is their authoritative list of crises they assess. The 17 orphan iso3 codes are countries that have FTS funding plans but are simply NOT in INFORM's monitoring universe:

- **Group A** (crisis-adjacent, not severe enough for INFORM): PRK, ZWE, ZMB, COG, BOL, VNM, NPL, RWA, MNG, LSO
- **Group B** (Ukraine refugee-hosting EU/EE states): POL, MDA, CZE, ROU, BGR, LTU, EST

**Root cause:** These countries are absent from the INFORM **Crisis Registry** (the curated "Lists" sheet of 127 actively monitored crises), NOT from the INFORM severity assessment itself. They DO appear in the `silver INFORM all_crises` table with real severity scores (e.g., PRK=4.1, ZWE=3.7, ZMB=2.8, COG=2.3, POL=1.6, BGR=1.5). The `severity_norm` is populated from these real assessments and flows correctly into the scoring formula.

**What is actually missing:** Registry-derived metadata fields:
- `country_name` (NULL — no registry entry to source it from)
- `crisis_drivers` (unavailable)
- `severity_trend_label` (defaults to NULL → `trend_multiplier = 1.0`)
- `region` assignment from the registry (falls back to C&T Taxonomy)

**Impact:** 50 rows in Table 3 (13.3%) have `inform_tracked = FALSE`. Their `overlooked_score` uses all components including a valid `severity_norm` (range [0.24, 0.82], mean=0.47). However, `trend_multiplier` defaults to 1.0 (neutral) since no trend label is available. Scores are fully valid but lack crisis context metadata.

**Mitigation:** The `inform_tracked` boolean column + Genie guardrail instruction ensures the RAG agent notes the absence of registry context. The C&T Taxonomy provides region and country_name fallback for display purposes.

## V1 Gap Formula Evaluation: Current Multiplicative vs. Two-Stage Approach

**Date:** 2025-05-23  
**Method:** Empirical evaluation across 8 criteria using 377 scored rows (63 countries in 2025, 71 total across 2020–2026)

---

### Formulas Compared

| | Current (6-factor multiplicative) | Proposed Two-Stage |
|---|---|---|
| **Score** | severity_norm × funding_gap_norm × scale_norm × trend_multiplier × persistence_multiplier × allocation_neglect | Gap Score = severity_norm × funding_gap_norm × scale_norm |
| **Ranking** | Sort by score DESC | Sort by Ranking Signal (trend × persistence) DESC, then Gap Score as tiebreaker |
| **Semantic model** | Trend/persistence are *amplifiers* of an existing gap | Trend/persistence are *primary discriminators* independent of gap magnitude |

---

### Evaluation Results (2025, n=63)

| Criterion | Current | Two-Stage | Winner |
|---|---|---|---|
| Score discrimination (P90/P10) | 4.83x | N/A (ranking signal has only 16 discrete values) | Current |
| Face validity (8 known severe crises in top 20) | 5/8 | 1/8 | Current |
| Tie problem | 0 ties | 93.7% of countries in tie groups | Current |
| Edge case: Philippines (147% funded) | Rank #61 (correct) | Rank #8 (incorrect) | Current |
| Stable catastrophes (SDN, AFG, SSD, COD, SYR) | Ranks 9–30 | Ranks 43–62 | Current |
| Rank correlation with current (Spearman ρ, avg 2020–2026) | — | ρ=0.41 (signal-first) | Current |
| Conceptual alignment with "overlooked" definition | Amplifiers: can't inflate non-crises | Priority sort: promotes worsening over catastrophic | Current |

---

### Critical Failures of Two-Stage Approach

**1. Stable Catastrophe Paradox**  
Crises that have been severe for so long they've stabilised (Sudan severity=4.7, DRC=4.5, Afghanistan=4.5) receive ranking_signal=1.0 (Stable trend). They rank BELOW Ecuador (severity=2.8) and Dominican Republic (severity=2.0) which have Increasing trend + high persistence. This inverts the meaning of "overlooked" — the most neglected crises are precisely the stable ones the world has stopped paying attention to.

**2. Over-Funded Crisis Promotion**  
Philippines (147.6% funded, gap_score=0.0) ranks #8 under signal-first because ranking_signal=1.56 (Increasing trend × persistence). The primary sort ignores the gap entirely when the ranking signal is high.

**3. Discretisation Collapse**  
The ranking signal can only take ~16 values (combinations of 4 trend levels × 6 persistence levels). With 63 countries, 93.7% fall into tie groups of 2–12. Within ties, gap_score becomes the de-facto sort — making the "two-stage" approach collapse back into gap_score ordering with occasional disruptions.

---

### Why the Current Formula Works

The multiplicative model is semantically correct for the question "which crises are most overlooked":

- **Trend** says: "A bad situation getting worse deserves more attention" — but cannot promote a non-crisis (1.3 × 0 = 0)
- **Persistence** says: "A chronic problem is more overlooked than a new one" — but only if the underlying gap exists
- **All factors must be simultaneously present** for a high rank: severe AND underfunded AND large-scale

The two-stage approach breaks this by allowing trend/persistence to override the gap measurement entirely.

---

### Verdict

**Retain the current multiplicative formula.** The two-stage approach fails the most important test: it does not identify the most overlooked crises. Instead it identifies the most *worsening* crises regardless of actual need-funding mismatch.

---

### Spearman Rank Correlation (Current vs. Alternatives, by Year)

| Year | N | vs Gap Score Only | vs Two-Stage Composite | vs Signal-First |
|---|---|---|---|---|
| 2020 | 47 | ρ=0.934 | ρ=0.997 | ρ=0.637 |
| 2021 | 49 | ρ=0.946 | ρ=0.998 | ρ=0.544 |
| 2022 | 55 | ρ=0.931 | ρ=1.000 | ρ=0.486 |
| 2023 | 61 | ρ=0.918 | ρ=0.999 | ρ=0.466 |
| 2024 | 65 | ρ=0.952 | ρ=1.000 | ρ=0.253 |
| 2025 | 63 | ρ=0.920 | ρ=0.999 | ρ=0.303 |
| 2026 | 37 | ρ=0.890 | ρ=0.976 | ρ=0.154 |

The signal-first ranking diverges dramatically from the current formula (ρ < 0.5 in most years), confirming it measures a fundamentally different concept.

%md
##User Msg to Hand over Context to a New Chat: 
I want to move to a new chat to design the upstream layers. Generate a prompt for the new chat that has all the necessary context from this chat like the goal, the plan, what is implemented, decisions, gaps, open items, etc. Think through your process step by step before generating a message
